# 04 — MLflow Judges & LLM Evaluation

**UI tab:** Evaluation runs · Judges

MLflow 3.x uses `mlflow.genai.evaluate()` with **scorers** — functions that
score model outputs, either via code or via an LLM-as-a-Judge.

```
eval_dataset  →  predict_fn  →  outputs
outputs       →  scorers     →  scores per row
scores        →  MLflow      →  Evaluation run (Evaluation runs tab)
```

> Start the MLflow server first: `mlflow server --host 127.0.0.1 --port 5000`

In [ ]:
!pip install mlflow google-genai pandas --quiet

In [ ]:
import os, re, json
import pandas as pd
from google import genai
from google.genai import types
import mlflow
from mlflow.genai import scorer

os.environ["GOOGLE_API_KEY"] = "YOUR_GOOGLE_API_KEY_HERE"
client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("04-MLflow-Judges")

print("MLflow", mlflow.__version__, "ready")

## Step 1 — Evaluation dataset

In [ ]:
# Format: list of dicts with 'inputs' and 'expectations' keys
# predict_fn receives inputs as kwargs and fills in 'outputs'
eval_dataset = [
    {
        "inputs": {"question": "What is MLflow?"},
        "expectations": {"expected_response": "MLflow is an open-source platform for managing the end-to-end ML lifecycle."},
    },
    {
        "inputs": {"question": "What is experiment tracking?"},
        "expectations": {"expected_response": "Experiment tracking records parameters, metrics and outputs for reproducibility."},
    },
    {
        "inputs": {"question": "What is a model registry?"},
        "expectations": {"expected_response": "A model registry is a central store for versioning and managing ML models."},
    },
]

# predict_fn: called once per row; 'question' matches the key in inputs
def predict(question: str) -> str:
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=[question],
        config=types.GenerateContentConfig(system_instruction="Answer in 1-2 sentences.")
    )
    return (response.text or '').strip()

print(f"Eval dataset: {len(eval_dataset)} rows ready")

## Step 2 — Define scorers

In [ ]:
# Scorer 1: LLM-as-a-Judge using Gemini Pro
@scorer
def gemini_correctness(inputs: dict, outputs: str, expectations: dict) -> bool:
    """Gemini Pro judges whether the answer is factually correct."""
    q = inputs.get("question", "")
    ground_truth = expectations.get("expected_response", "")
    prompt = (
        f"Is this answer factually correct?\n"
        f"Question: {q}\n"
        f"Correct answer: {ground_truth}\n"
        f"Given answer: {outputs}\n"
        f"Reply ONLY with true or false."
    )
    response = client.models.generate_content(
        model="gemini-2.5-pro",
        contents=[prompt],
        config=types.GenerateContentConfig(temperature=0.0, max_output_tokens=10)
    )
    text = (response.text or '').strip().lower()
    return "true" in text

# Scorer 2: Code-based, no LLM needed
@scorer
def conciseness(outputs: str) -> bool:
    """Passes if the answer is 30 words or fewer."""
    return len((outputs or '').split()) <= 30

print("Scorers ready: gemini_correctness, conciseness")

## Step 3 — Run evaluation

In [ ]:
# mlflow.genai.evaluate():
#   1. calls predict(question=...) for every row
#   2. runs each scorer on the outputs
#   3. logs an Evaluation Run → check the Evaluation runs tab
results = mlflow.genai.evaluate(
    data=eval_dataset,
    predict_fn=predict,
    scorers=[gemini_correctness, conciseness],
)

print("Evaluation complete.")
print("Open MLflow UI → Evaluation runs tab to see per-row scores.")
try:
    print(results.results.to_string())
except Exception:
    print(results)

## Step 4 — Judges tab

The **Judges** tab in MLflow UI is for registering judges that run **automatically**
on every incoming trace (continuous / production evaluation).

**Option A — via UI** *(easiest)*: click **+ New LLM judge** or **< > New custom code judge**
inside the Judges tab. Configure model + instructions there.

**Option B — via code** *(requires MLflow AI Gateway)*:
```python
from mlflow.genai.scorers import Correctness, ScorerSamplingConfig
# Uses LiteLLM internally: pip install litellm
judge = Correctness(model="gemini:/gemini-2.5-flash")
registered = judge.register(name="gemini_correctness")
registered.start(sampling_config=ScorerSamplingConfig(sample_rate=1.0))
```
Once registered, the judge scores every new trace automatically.

**Next →** `05_mlflow_datasets.ipynb`